In [ ]:
%pip install openai
%pip install requests
%pip install Pillow
%pip install ipywidgets

In [ ]:
import openai
from openai import OpenAI
import requests
import time
import random
from PIL import Image
from io import BytesIO
import concurrent.futures
from IPython.display import display, HTML, Image as IPImage
import ipywidgets as widgets
import base64

In [ ]:
# --- Configuration ---
# Add your API keys and Base URLs here
TEXT_API_CONFIG = {
    "api_key": "",
    "base_url": "https://openrouter.ai/api/v1"
}

IMAGE_API_CONFIG = {
    "api_key": "",
    "base_url": "https://openrouter.ai/api/v1/images" # e.g., https://api.openai.com/v1/images/generations
}

TEXT_MODELS = ["google/gemini-3.8-flash", "xiaomi/mimo-v2.5","minimax/minimax-m2.7"] # Add 2-3 models
IMAGE_MODELS = ["qwen/qwen-image-3-pro"]

In [ ]:
def get_ai_response(model_name, system_prompt, user_prompt):
    """Generates prompt with a small random delay to avoid rate limits."""
    time.sleep(random.uniform(1, 3)) # 1-3 second staggered start
    client = OpenAI(api_key=TEXT_API_CONFIG["api_key"], base_url=TEXT_API_CONFIG["base_url"])
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
            temperature=0.7
        )
        print(f"{model_name}", response.choices[0].message.content.strip())

        return {"model": model_name, "prompt": response.choices[0].message.content.strip()}
    except Exception as e:
        return {"model": model_name, "prompt": f"TEXT_ERROR: {str(e)}"}
    
def generate_image(prompt_data, image_model, retries=1):
    # INCREASED DELAY: Cloudflare is sensitive. 
    # We wait longer between each image request.
    time.sleep(random.uniform(8, 12)) 
    
    headers = {
        "Authorization": f"Bearer {IMAGE_API_CONFIG['api_key']}", 
        "Content-Type": "application/json"
    }
    payload = {
        "model": image_model, 
        "prompt": prompt_data["prompt"], 
        "n": 1, 
        "size": "1024x1024"
    }
    
    last_error = ""
    for attempt in range(retries + 1):
        try:
            resp = requests.post(IMAGE_API_CONFIG["base_url"], headers=headers, json=payload, timeout=90)
            
            if resp.status_code == 429 or resp.status_code == 1015:
                last_error = "Rate Limited by Cloudflare. Waiting longer..."
                time.sleep(20) # Wait 20 seconds if limited
                continue

            if resp.status_code != 200:
                last_error = f"Status {resp.status_code}: {resp.text}"
                continue

            data = resp.json()
            if 'data' not in data:
                last_error = f"API Message: {data.get('error', {}).get('message', 'No data found')}"
                continue

            img_url = data['data'][0]['url']
            img_data = requests.get(img_url, timeout=30).content
            return {
                "text_model": prompt_data["model"], 
                "img_model": image_model, 
                "used_prompt": prompt_data["prompt"], 
                "data": img_data
            }
        except Exception as e:
            last_error = str(e)
            time.sleep(10)
            
    return {
        "text_model": prompt_data["model"], 
        "img_model": image_model, 
        "used_prompt": prompt_data["prompt"], 
        "error": last_error
    }

In [ ]:
def display_results_robust(results):
    html_output = '<div style="display: flex; flex-wrap: wrap; gap: 20px; justify-content: center;">'
    for i, res in enumerate(results):
        card_style = "border: 1px solid #ddd; padding: 15px; border-radius: 12px; width: 450px; background: #1e1e1e; color: #fff; box-shadow: 0 4px 8px rgba(0,0,0,0.2);"
        html_output += f'<div style="{card_style}">'
        
        if "error" in res:
            # Enhanced Error Display
            html_output += f"""
                <div style="background: #3d1414; padding: 15px; border-radius: 8px; border-left: 5px solid #ff4444;">
                    <strong style="color: #ff4444;">Generation Failed</strong><br>
                    <small><b>Models:</b> {res['text_model']} + {res['img_model']}</small><br>
                    <div style="margin-top:10px; font-family: monospace; font-size: 0.8em; overflow-x: auto;">
                        <b>Reason:</b> {res['error']}
                    </div>
                </div>
            """
        else:
            b64_img = base64.b64encode(res['data']).decode('ascii')
            filename = f"gen_{res['text_model']}_{res['img_model']}.png"
            html_output += f'<img src="data:image/png;base64,{b64_img}" style="width: 100%; border-radius: 8px;"/>'
            html_output += f"""
                <div style="margin-top: 12px;">
                    <p style="margin: 5px 0;"><b>{res['text_model']}</b> <span style="color: #888;">via</span> <b>{res['img_model']}</b></p>
                    <details style="margin-bottom: 10px; opacity: 0.8;"><summary>View Prompt</summary>
                        <div style="background: #2d2d2d; padding: 8px; font-size: 0.8em; max-height: 80px; overflow-y: auto;">{res['used_prompt']}</div>
                    </details>
                    <a href="data:image/png;base64,{b64_img}" download="{filename}" 
                       style="display: block; text-align: center; background: #28a745; color: white; padding: 10px; text-decoration: none; border-radius: 6px; font-weight: bold;">
                       Save Full Resolution
                    </a>
                </div>
            """
        html_output += '</div>'
    html_output += '</div>'
    display(HTML(html_output))

In [ ]:
def run_pipeline(hindi_text):
    system_msg = "You are an expert AI art director specializing in Hindi literature and visual storytelling."
    user_query = f"""
    I will provide you with a piece of Hindi prose. You will analyze the emotion, the metaphors, and the atmosphere of the text. Then, create a highly detailed image generation prompt which will portray a similar but not same feel as the prose.

    Guidelines for the Prompt:

        Values: Focus on nature, simplicity, truth, and dignity.

        Avoid: No vulgarity, no modern/flashy aesthetics, and no chaotic designs.

        Style: Prefer styles like "Realistic Oil Painting," "Soft Watercolor," or "Detailed Cinematic Photography with natural light."

        Output: Provide only the English image prompt and nothing else that I can use in an image generator.
    Text: {hindi_text}
    """

    # 1. Generate Text Prompts Parallely
    # print(f"--- Generating prompts from {len(TEXT_MODELS)} models... ---")
    # with concurrent.futures.ThreadPoolExecutor() as executor:
    #     prompt_results = list(executor.map(
    #         lambda m: get_ai_response(m, system_msg, user_query), 
    #         TEXT_MODELS
    #     ))

    # 2. Generate Images Parallely (Each prompt goes to each image model)
    # print(f"--- Generating images from {len(IMAGE_MODELS)} models... ---")
    # jobs = []
    # with concurrent.futures.ThreadPoolExecutor() as executor:
    #     for p_data in prompt_results:
    #         if "Error" in p_data["prompt"]: continue
    #         for i_model in IMAGE_MODELS:
    #             jobs.append(executor.submit(generate_image, p_data, i_model))
        
    #     image_results = [job.result() for job in concurrent.futures.as_completed(jobs)]
        
    # if not image_results:
    #     print("No images were generated. Check your API keys and Base URLs.")
    # else:
    #     print(f"--- Successfully generated {len(image_results)} results. Rendering... ---")
    #     display_results_robust(image_results)

    with concurrent.futures.ThreadPoolExecutor() as executor:
    #     # Prompt stage
        p_results = list(executor.map(lambda m: get_ai_response(m, system_msg, user_query), TEXT_MODELS))
        
    # Image stage
    results = []
    # CHANGED: We now run images one-by-one to prevent Cloudflare Error 1015
    for p in p_results:
        if "TEXT_ERROR" in p["prompt"]: continue
        for im in IMAGE_MODELS:
            print(f"🎨 Requesting image from {im} using {p['model']} prompt...")
            res = generate_image(p, im)
            results.append(res)
    
    display_results_robust(results)



In [ ]:
run_pipeline("""

हुआ तो था...
शायद...
दर्द... ना तो...
पर... 
समझना नहीं है...
सुनना... नहीं है...
सूखे पत्तों पर...
तेरे क़दमों की... आहट...
एक बार... फिर से...
वही कसक...
ना... अब नहीं...
प्रतीक्षा... और‌ नहीं...
पाषाण... दरकता रहे...
टूटती रहें...
ख़्वाहिशें...



""")

In [ ]:
run_pipeline("""

हुआ तो था...
शायद...
दर्द... ना तो...
पर... 
समझना नहीं है...
सुनना... नहीं है...
सूखे पत्तों पर...
तेरे क़दमों की... आहट...
एक बार... फिर से...
वही कसक...
ना... अब नहीं...
प्रतीक्षा... और‌ नहीं...
पाषाण... दरकता रहे...
टूटती रहें...
ख़्वाहिशें...


""")

In [ ]:
"""
    I will provide you with a piece of Hindi prose. You will analyze the emotion, the metaphors, and the atmosphere of the text. Then, you will create a highly detailed image generation prompt which will portray not same but similar feel as the prose.

    Guidelines for the Prompt:

        Values: Focus on nature, simplicity, truth, and dignity.

        Avoid: No vulgarity, no modern/flashy aesthetics, and no chaotic designs.

        Style: Prefer styles like "Realistic Oil Painting," "Soft Watercolor," or "Detailed Cinematic Photography with natural light."

        Key Elements: Always include descriptions of the lighting (e.g., golden hour, misty dawn) and the specific textures (e.g., weathered wood, silver hair, soft cotton).

        Output: Provide only the English image prompt and nothing else that I can use in an image generator.
    Text: {hindi_text}
    """

Matches 4 of the 7 required qualifications:

✓ Bachelor’s degree in Computer Science or a related technical field.

✓ Strong programming skills in Python and/or JavaScript/TypeScript.

✓ 3+ years of experience building and shipping production software; 2+ years working with LLMs or AI APIs.

✓ Strong communication skills and experience leading technical discussions with customers or partners.

? Hands-on experience with modern LLMs (e.g., GPT, Claude, Gemini), vector databases, and agent/orchestration frameworks (e.g., LangChain, LangGraph, LlamaIndex, or custom solutions). (No specific mention of LLMs or vector databases)

? Practical experience with RAG, agent workflows, evaluation, and performance optimization. (No specific mention of RAG or agent workflows)

? Strong agent design skills, including prompt engineering, tool use, multi-step agent workflows (e.g. ReAct), and failure handling. (No specific mention of agent design skills)

 

Matches 4 of the 6 additional qualifications:

✓ Experience working in a fast-moving startup environment.

✓ Prior work delivering AI or automation solutions to enterprise customers.

✓ Experience with cloud deployment and production operations for AI systems.

✓ Strong written and verbal communication skills.

? Familiarity with human-in-the-loop workflows, fine-tuning, or LLM evaluation techniques. (No specific mention of these techniques)

? Background in applied ML, NLP, or decision systems. (No specific mention of NLP or decision systems)

 

There are qualifications that will likely be evaluated in the application or interview:

• Ability to reason about and balance trade-offs between customization and reuse, as well as autonomy, control, cost, latency, and risk.

• Ability and interest to travel up to 25%, flexible.